In [ ]:
import pandas as pd
import numpy as np

import tensorflow
from sklearn.model_selection import train_test_split
import sklearn.decomposition as decomposition
from sklearn.manifold import TSNE
from sklearn.svm import SVC
from sklearn.metrics import f1_score, precision_recall_curve, auc, accuracy_score, roc_auc_score, confusion_matrix

from xgboost import XGBClassifier

from tensorflow.keras import Model
from tensorflow.keras.models import load_model

### Load MIT dataset

In [ ]:
df_train = pd.read_csv("data/mitbih_train.csv", header=None)
df_train = df_train.sample(frac=1)
df_test = pd.read_csv("data/mitbih_test.csv", header=None)

Y = np.array(df_train[187].values).astype(np.int8)
X = np.array(df_train[list(range(187))].values)[..., np.newaxis]

Y_test = np.array(df_test[187].values).astype(np.int8)
X_test = np.array(df_test[list(range(187))].values)[..., np.newaxis]

### Get embeddings from baseline model

In [ ]:
model = load_model("baseline_cnn_mitbih.h5")
embedding_model = Model(inputs=model.inputs, outputs=model.layers[15].output) # get embeddings from last conv layer

embeddings = embedding_model.predict(X)
embeddings_test = embedding_model.predict(X_test)
embeddings.shape

### SVM (takes a while)

In [ ]:
svm = SVC()
svm.fit(embeddings, Y)

In [ ]:
pred_test = svm.predict(embeddings_test)

f1 = f1_score(Y_test, pred_test, average="macro")

print("Test f1 score : %s "% f1)

acc = accuracy_score(Y_test, pred_test)

print("Test accuracy score : %s "% acc)

print(confusion_matrix(Y_test, pred_test))

### Try applying pca to embeddings (much faster)

In [ ]:
embeddings_normalized = embeddings - embeddings.mean(axis=0)
embeddings_test_normalized = embeddings_test - embeddings_test.mean(axis=0)
pca = decomposition.PCA(n_components=64)
components = pca.fit_transform(embeddings_normalized)
components_test = pca.transform(embeddings_test_normalized)
svm_pca = SVC()
svm_pca.fit(components, Y)

In [ ]:
pred_test = svm_pca.predict(components_test)

f1 = f1_score(Y_test, pred_test, average="macro")

print("Test f1 score : %s "% f1)

acc = accuracy_score(Y_test, pred_test)

print("Test accuracy score : %s "% acc)

print(confusion_matrix(Y_test, pred_test))

### Forest

In [ ]:
clf = XGBClassifier(objective='multi:softmax', n_jobs=-1)
clf.fit(embeddings, Y)

In [ ]:
pred_test = clf.predict(embeddings_test)

f1 = f1_score(Y_test, pred_test, average="macro")

print("Test f1 score : %s "% f1)

acc = accuracy_score(Y_test, pred_test)

print("Test accuracy score : %s "% acc)

print(confusion_matrix(Y_test, pred_test))

### Try with PCA

In [ ]:
clf_pca = XGBClassifier(objective='multi:softmax', n_jobs=-1)
clf_pca.fit(components, Y)

In [ ]:
pred_test = svm_pca.predict(components_test)

f1 = f1_score(Y_test, pred_test, average="macro")

print("Test f1 score : %s "% f1)

acc = accuracy_score(Y_test, pred_test)

print("Test accuracy score : %s "% acc)

print(confusion_matrix(
    Y_test, pred_test))